## Iteration Changelog
- Updated inference simulation examples to current stage feature contract.
- Added board-risk and stack-aware scenario checks.
- Open issues: wire these checks into automated regression tests.

In [ ]:
from pathlib import Path

import pandas as pd

from scripts.features.poker_hand_strength import build_stage_feature_payload
from scripts.models.stage_win_predictor import predict_stage_win_probability

ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
print(ROOT)

## Scenario Checks

We simulate stage-level game states and verify predictions under stack pressure and dangerous shared-board textures.

In [ ]:
# Safe board scenario
safe_features = build_stage_feature_payload(
    "turn",
    ["Ah", "Kd"],
    ["Qs", "7d", "2c", "9h"],
    total_bet=45,
    current_pot=80,
    hero_stack=120,
    table_stacks=[120, 95, 160],
    big_blind=2,
)

# Risky shared-board scenario (4-flush board)
risky_features = build_stage_feature_payload(
    "turn",
    ["As", "Kd"],
    ["Qh", "Jh", "2h", "9h"],
    total_bet=45,
    current_pot=80,
    hero_stack=120,
    table_stacks=[120, 95, 160],
    big_blind=2,
)

safe_p = predict_stage_win_probability("turn", safe_features)
risky_p = predict_stage_win_probability("turn", risky_features)

print("safe_p:", round(safe_p, 4), "safe_risk:", round(safe_features["board_shared_strength_risk"], 3))
print("risky_p:", round(risky_p, 4), "risky_risk:", round(risky_features["board_shared_strength_risk"], 3))

## Interpretation
- Compare risk score and probability jointly.
- A higher shared-board risk should push betting posture more cautious even when raw probability is similar.